# UK Turnout Analysis Project

## NB02: Census 2021 API Data

This notebook uses the ONS Census 2021 API to retrieve demographic and 
socioeconomic indicators for the 2024 election constituencies in England & Wales. 

My aim is to construct predictors of 2024 general election turnout, then merge this with the cleaned NB01 election dataset. Returned constituency codes are validated against the cleaned election dataset prior to merging. 

### 1. Using ONS API to retrieve geographical data

In [1]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.beta.ons.gov.uk/v1"

In [2]:
url = f"{BASE_URL}/population-types/UR/area-types"

response = requests.get(url, timeout=30)
response.raise_for_status()

area_types = response.json()

In [3]:
area_types.keys()

dict_keys(['limit', 'offset', 'count', 'total_count', 'items'])

In [4]:
area_types

{'limit': 30,
 'offset': 0,
 'count': 20,
 'total_count': 20,
 'items': [{'id': 'nat',
   'label': 'England and Wales',
   'description': 'Data for both England and Wales.',
   'total_count': 1,
   'hierarchy_order': 1400},
  {'id': 'ctry',
   'label': 'Countries',
   'description': 'Data for either the whole of England or Wales.',
   'total_count': 2,
   'hierarchy_order': 1300},
  {'id': 'rgn',
   'label': 'Regions',
   'description': 'Data for the nine regions in England, and Wales as a whole.',
   'total_count': 10,
   'hierarchy_order': 1200},
  {'id': 'lep',
   'label': 'Local enterprise partnerships',
   'description': 'Local enterprise partnerships (LEPs) are voluntary partnerships between local authorities and local private sector businesses to help determine local economic priorities, drive economic growth and job creation.',
   'total_count': 37,
   'hierarchy_order': 1150},
  {'id': 'nhser',
   'label': 'NHS England regions',
   'description': 'Each NHS region is responsibl

The post-2019 Westminster Parliamentary constituency geography (`p19wpc`)
corresponds to the constituencies used for the 2024 general election.

ONS API limits area responses to 500 records, so the 575 constituencies are
retrieved using pagination before their geographic codes are validated against
the NB01 election dataset.

### 2. Build the Census predictor dataset

Census data is retrieved programmatically using the ONS Census API.

First, the post-2019 Westminster Parliamentary constituency geography is identified and validated against the 2024 election dataset using official constituency codes. Demographic and socioeconomic variables are then retrieved at the same geographic level and transformed into constituency-level predictors for later modelling.

In [5]:
def get_all_areas(population_type, area_type, limit=500):
    url = (
        f"{BASE_URL}/population-types/"
        f"{population_type}/area-types/{area_type}/areas"
    )

    areas = []
    offset = 0

    while True:
        response = requests.get(
            url,
            params={"limit": limit, "offset": offset},
            timeout=30
        )
        response.raise_for_status()

        data = response.json()
        areas.extend(data["items"])

        offset += data["count"]

        if offset >= data["total_count"]:
            break

    return areas

In [6]:
def flatten_census_response(data, dimension_id, value_name):
    rows = []

    for item in data["observations"]:
        dimensions = {
            dim["dimension_id"]: dim
            for dim in item["dimensions"]
        }

        rows.append({
            "constituency_code": dimensions["p19wpc"]["option_id"],
            "constituency_name": dimensions["p19wpc"]["option"],
            f"{value_name}_code": dimensions[dimension_id]["option_id"],
            value_name: dimensions[dimension_id]["option"],
            "count": item["observation"]
        })

    return pd.DataFrame(rows)

In [7]:
constituencies = get_all_areas("UR", "p19wpc")

len(constituencies)

575

In [8]:
constituencies_df = (
    pd.DataFrame(constituencies)[["id", "label"]]
    .rename(columns={
        "id": "constituency_code",
        "label": "constituency_name"
    })
)

constituencies_df.head()

,constituency_code,constituency_name
0,E14001063,Aldershot
1,E14001064,Aldridge-Brownhills
2,E14001065,Altrincham and Sale West
3,E14001066,Amber Valley
4,E14001067,Arundel and South Downs


In [9]:
assert len(constituencies_df) == 575
assert constituencies_df["constituency_code"].is_unique
assert constituencies_df["constituency_code"].notna().all()

In [10]:
election = pd.read_csv(
    "../data/processed/election_results_2024.csv"
)

election_codes = set(election["constituency_code"])
ons_codes = set(constituencies_df["constituency_code"])

missing_from_ons = election_codes - ons_codes
missing_from_election = ons_codes - election_codes

print("Election codes missing from ONS:", missing_from_ons)
print("ONS codes missing from election data:", missing_from_election)

Election codes missing from ONS: set()
ONS codes missing from election data: set()


In [11]:
assert election_codes == ons_codes

In [12]:
url = f"{BASE_URL}/population-types/UR/dimensions"

response = requests.get(
    url,
    params={"q": "qualification"},
    timeout=30
)
response.raise_for_status()

qualification_dimensions = response.json()
qualification_dimensions

{'limit': 30,
 'offset': 0,
 'count': 1,
 'total_count': 44,
 'items': [{'id': 'highest_qualification',
   'label': 'Highest level of qualification (8 categories)',
   'description': 'The highest level of qualification is derived from the question asking people to indicate all qualifications held, or their nearest equivalent.\n\nThis may include foreign qualifications where they were matched to the closest UK equivalent.',
   'total_count': 8,
   'quality_statement_text': 'There are quality considerations about higher education qualifications, including those at Level 4+, responses from older people and international migrants, and comparability with 2011 Census data.'}]}

### 3. Retrieving highest-qualification observations

Highest qualification is used as the first socioeconomic predictor. The API
request returns the qualification categories for all post-2019 Westminster
constituencies in England and Wales.

In [13]:
url = f"{BASE_URL}/population-types/UR/census-observations"

params = {
    "area-type": "p19wpc",
    "dimensions": "highest_qualification"
}

response = requests.get(
    url,
    params=params,
    timeout=60
)

response.raise_for_status()

qualification_data = response.json()

In [14]:
qualification_data.keys()

dict_keys(['observations', 'links', 'total_observations', 'blocked_areas', 'total_areas', 'areas_returned'])

In [15]:
print("Total observations:", qualification_data["total_observations"])
qualification_data["observations"][:3]

Total observations: 4600


[{'dimensions': [{'dimension': 'Post-2019 Westminster Parliamentary constituencies',
    'dimension_id': 'p19wpc',
    'option': 'Aldershot',
    'option_id': 'E14001063'},
   {'dimension': 'Highest level of qualification (8 categories)',
    'dimension_id': 'highest_qualification',
    'option': 'Does not apply',
    'option_id': '-8'}],
  'observation': 21769},
 {'dimensions': [{'dimension': 'Post-2019 Westminster Parliamentary constituencies',
    'dimension_id': 'p19wpc',
    'option': 'Aldershot',
    'option_id': 'E14001063'},
   {'dimension': 'Highest level of qualification (8 categories)',
    'dimension_id': 'highest_qualification',
    'option': 'No qualifications',
    'option_id': '0'}],
  'observation': 16899},
 {'dimensions': [{'dimension': 'Post-2019 Westminster Parliamentary constituencies',
    'dimension_id': 'p19wpc',
    'option': 'Aldershot',
    'option_id': 'E14001063'},
   {'dimension': 'Highest level of qualification (8 categories)',
    'dimension_id': 'highes

In [16]:
assert qualification_data["total_observations"] == 4600
assert qualification_data["total_areas"] == 575
assert qualification_data["areas_returned"] == 575

In [17]:
qualification_long = flatten_census_response(
    qualification_data,
    dimension_id="highest_qualification",
    value_name="qualification"
)

In [18]:
qualification_long[
    ["qualification_code", "qualification"]
].drop_duplicates().sort_values("qualification_code")

,qualification_code,qualification
0,-8,Does not apply
1,0,No qualifications
2,1,Level 1 and entry level qualifications: 1 to 4...
3,2,Level 2 qualifications: 5 or more GCSEs (A* to...
4,3,Apprenticeship
5,4,Level 3 qualifications: 2 or more A levels or ...
6,5,"Level 4 qualifications or above: degree (BA, B..."
7,6,Other: vocational or work-related qualificatio...


In [19]:
qualification_long.groupby("constituency_code").size().value_counts()

8    575
Name: count, dtype: int64

#### Construct the Level 4+ qualification predictor

The highest-qualification variable includes a `Does not apply` category for
people outside the applicable population, such as under 15s. This category is excluded from the denominator.

Our predictor measures the percentage of applicable residents - `Does not apply` excluded -  whose highest qualification is Level 4 or above.

In [20]:
applicable_qualifications = qualification_long[
    qualification_long["qualification_code"] != "-8"
].copy()

qualification_totals = (
    applicable_qualifications
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("qualification_total")
)

level4_counts = (
    applicable_qualifications[
        applicable_qualifications["qualification_code"] == "5"
    ]
    .set_index(["constituency_code", "constituency_name"])["count"]
    .rename("level4_plus_count")
)

qualification_features = (
    pd.concat([qualification_totals, level4_counts], axis=1)
    .reset_index()
)

qualification_features["level4_plus_pct"] = (
    qualification_features["level4_plus_count"]
    / qualification_features["qualification_total"]
    * 100
)

In [21]:
qualification_features.head()

,constituency_code,constituency_name,qualification_total,level4_plus_count,level4_plus_pct
0,E14001063,Aldershot,95661,28644,29.943237
1,E14001064,Aldridge-Brownhills,76953,20894,27.151638
2,E14001065,Altrincham and Sale West,79446,38655,48.655691
3,E14001066,Amber Valley,75007,18245,24.324396
4,E14001067,Arundel and South Downs,81759,30616,37.446642


In [22]:
assert len(qualification_features) == 575
assert qualification_features["constituency_code"].is_unique
assert qualification_features["level4_plus_pct"].between(0, 100).all()

In [23]:
qualification_features["level4_plus_pct"].describe()

count    575.000000
mean      33.571435
std        9.020211
min       18.084787
25%       27.041060
50%       32.048674
75%       37.615543
max       65.221223
Name: level4_plus_pct, dtype: float64

In [24]:
qualification_features.nlargest(
    10, "level4_plus_pct"
)[
    ["constituency_name", "level4_plus_pct"]
]

,constituency_name,level4_plus_pct
18,Battersea,65.221223
202,Hampstead and Highgate,64.692245
382,Richmond Park,64.103058
109,Cities of London and Westminster,62.202815
487,Tooting,61.776277
371,Putney,60.890001
97,Chelsea and Fulham,60.582824
247,Kensington and Bayswater,60.555357
523,Wimbledon,60.510291
112,Clapham and Brixton Hill,60.293483


In [25]:
qualification_features.nsmallest(
    10, "level4_plus_pct"
)[
    ["constituency_name", "level4_plus_pct"]
]

,constituency_name,level4_plus_pct
51,Boston and Skegness,18.084787
111,Clacton,18.152828
193,Great Yarmouth,18.243693
91,Castle Point,18.504501
250,Kingston upon Hull East,18.606187
484,Tipton and Wednesbury,18.936525
327,North East Cambridgeshire,19.039985
192,Great Grimsby and Cleethorpes,19.560980
12,Barnsley South,20.192014
32,Birmingham Hodge Hill and Solihull North,20.447997


### 4. Retrieve age-structure observations

Age structure is included as a demographic predictor of turnout.

An age classification that can be aggregated into interpretable constituency-level measures, such as the share of residents aged 18–34 and the share aged 65 and over, is requested from the ONS API.

In [26]:
url = f"{BASE_URL}/population-types/UR/dimensions"

response = requests.get(
    url,
    params={"q": "age"},
    timeout=30
)
response.raise_for_status()

age_dimensions = response.json()
age_dimensions

{'limit': 30,
 'offset': 0,
 'count': 7,
 'total_count': 44,
 'items': [{'id': 'age_arrival_uk_23a',
   'label': 'Age of arrival in the UK (23 categories)',
   'description': 'The date a person last arrived to live in the UK and their age. Arrival dates do not include returning from short trips away from the UK.\n\nAge of arrival only applies to usual residents not born in the UK. It does not include usual residents born in the UK who have emigrated and since returned. These are recorded in the category “born in the UK”.',
   'total_count': 23,
   'quality_statement_text': ''},
  {'id': 'english_proficiency',
   'label': 'Proficiency in English language (6 categories)',
   'description': 'How well people whose main language is not English (English or Welsh in Wales) speak English.',
   'total_count': 6,
   'quality_statement_text': ''},
  {'id': 'main_language_23a',
   'label': 'Main language (23 categories)',
   'description': 'A person\'s first or preferred language. \n\nThis breaks 

#### Identifying an appropriate age classification

The dataset's age variable contains 101 single-year categories. Rather than
using individual ages directly, available ONS age groupings are inspected to
identify a classification suitable for constructing an interpretable age predictor.

In [27]:
age_dimension = "resident_age_101a"

url = (
    f"{BASE_URL}/population-types/UR/"
    f"dimensions/{age_dimension}/categorisations"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

age_categorisations = response.json()

In [28]:
for cat in age_categorisations["items"]:
    print(
        "\nID:", cat["id"],
        "|", cat["label"]
    )
    for category in cat["categories"]:
        print("  ", category["id"], "-", category["label"])


ID: resident_age_2a | Age (2 categories)
   1 - Aged 49 years and under
   2 - Aged 50 years and over

ID: resident_age_3a | Age (3 categories)
   1 - Aged 15 years and under
   2 - Aged 16 to 64 years
   3 - Aged 65 years and over

ID: resident_age_3b | Age (B) (3 categories)
   1 - Aged 15 years and under
   2 - Aged 16 to 49 years
   3 - Aged 50 years and over

ID: resident_age_3c | Age (C) (3 categories)
   1 - Aged 24 years and under
   2 - Aged 25 to 49 years
   3 - Aged 50 years and over

ID: resident_age_3d | Age (D) (3 categories)
   1 - Aged 64 years and under
   2 - Aged 65 to 84 years
   3 - Aged 85 years and over

ID: resident_age_4a | Age (4 categories)
   1 - Aged 15 years and under
   2 - Aged 16 to 20 years
   3 - Aged 21 to 29 years
   4 - Aged 30 years and over

ID: resident_age_4b | Age (B) (4 categories)
   1 - Aged 15 years and under
   2 - Aged 16 to 49 years
   3 - Aged 50 to 64 years
   4 - Aged 65 years and over

ID: resident_age_4c | Age (C) (4 categories)
 

In [29]:
age_categorisations["items"][:3]

[{'id': 'resident_age_2a',
  'label': 'Age (2 categories)',
  'categories': [{'id': '1', 'label': 'Aged 49 years and under'},
   {'id': '2', 'label': 'Aged 50 years and over'}],
  'quality_statement_text': 'Estimates for single year of age between ages 90 and 100+ are less reliable than other ages. Estimation and adjustment at these ages was based on the age range 90+ rather than five-year age bands.',
  'default_categorisation': False},
 {'id': 'resident_age_3a',
  'label': 'Age (3 categories)',
  'categories': [{'id': '1', 'label': 'Aged 15 years and under'},
   {'id': '2', 'label': 'Aged 16 to 64 years'},
   {'id': '3', 'label': 'Aged 65 years and over'}],
  'quality_statement_text': 'Estimates for single year of age between ages 90 and 100+ are less reliable than other ages. Estimation and adjustment at these ages was based on the age range 90+ rather than five-year age bands.',
  'default_categorisation': False},
 {'id': 'resident_age_3b',
  'label': 'Age (B) (3 categories)',
  'c

#### Selected age classification

The `resident_age_14a` categorisation is used, as it separates people aged
16–17 from the voting-age population and provides sufficiently detailed age
bands to construct exact measures for residents aged 18–34 and 65 and over.

Both measures are calculated as percentages of the population aged 18 and over.

In [30]:
age_dimension = "resident_age_14a"

url = f"{BASE_URL}/population-types/UR/census-observations"

params = {
    "area-type": "p19wpc",
    "dimensions": age_dimension
}

response = requests.get(
    url,
    params=params,
    timeout=60
)

response.raise_for_status()

age_data = response.json()

In [31]:
print("Total observations:", age_data["total_observations"])
print("Total areas:", age_data["total_areas"])
print("Areas returned:", age_data["areas_returned"])

assert age_data["total_observations"] == 575 * 14
assert age_data["total_areas"] == 575
assert age_data["areas_returned"] == 575

Total observations: 8050
Total areas: 575
Areas returned: 575


In [32]:
age_long = flatten_census_response(
    age_data,
    dimension_id="resident_age_14a",
    value_name="age_group"
)

age_long.head()

,constituency_code,constituency_name,age_group_code,age_group,count
0,E14001063,Aldershot,1,Aged 15 years and under,21769
1,E14001063,Aldershot,2,Aged 16 to 17 years,2687
2,E14001063,Aldershot,3,Aged 18 to 19 years,2469
3,E14001063,Aldershot,4,Aged 20 to 21 years,2289
4,E14001063,Aldershot,5,Aged 22 to 24 years,4277


#### Construct age-structure predictors

Two age-structure measures are constructed using the population aged 18 and over as the denominator:

- the percentage aged 18–34;
- the percentage aged 65 and over.

Using the adult population as the denominator excludes residents who were not of voting age at the 2024 general election.

In [33]:
adult_age_codes = [str(i) for i in range(3, 15)]
young_age_codes = [str(i) for i in range(3, 8)]
older_age_codes = ["14"]

adult_population = (
    age_long[
        age_long["age_group_code"].isin(adult_age_codes)
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("adult_population")
)

young_population = (
    age_long[
        age_long["age_group_code"].isin(young_age_codes)
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("age_18_34_count")
)

older_population = (
    age_long[
        age_long["age_group_code"].isin(older_age_codes)
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("age_65_plus_count")
)

age_features = (
    pd.concat(
        [adult_population, young_population, older_population],
        axis=1
    )
    .reset_index()
)

age_features["age_18_34_pct"] = (
    age_features["age_18_34_count"]
    / age_features["adult_population"]
    * 100
)

age_features["age_65_plus_pct"] = (
    age_features["age_65_plus_count"]
    / age_features["adult_population"]
    * 100
)

In [34]:
assert len(age_features) == 575
assert age_features["constituency_code"].is_unique
assert age_features["age_18_34_pct"].between(0, 100).all()
assert age_features["age_65_plus_pct"].between(0, 100).all()

age_features[
    ["age_18_34_pct", "age_65_plus_pct"]
].describe()

,age_18_34_pct,age_65_plus_pct
count,575.000000,575.000000
mean,27.088076,23.855425
std,7.465308,6.387967
min,15.880229,6.235623
25%,22.188620,19.963960
50%,25.272725,24.350059
75%,29.770854,28.237907
max,66.517514,41.174165


In [35]:
age_features.nlargest(
    10, "age_18_34_pct"
)[
    ["constituency_name", "age_18_34_pct"]
]

,constituency_name,age_18_34_pct
256,Leeds Central and Headingley,66.517514
404,Sheffield Central,61.677794
68,Bristol Central,59.093003
290,Manchester Rusholme,57.806641
275,Liverpool Riverside,52.717359
23,Bethnal Green and Stepney,51.700083
367,Poplar and Limehouse,51.363417
33,Birmingham Ladywood,50.629233
349,Nottingham South,49.528591
112,Clapham and Brixton Hill,49.351293


In [36]:
age_features.nlargest(
    10, "age_65_plus_pct"
)[
    ["constituency_name", "age_65_plus_pct"]
]

,constituency_name,age_65_plus_pct
108,Christchurch,41.174165
333,North Norfolk,40.426827
311,New Forest West,39.644970
25,Bexhill and Battle,38.720157
111,Clacton,38.572515
512,West Dorset,38.530095
228,Honiton and Sidmouth,38.249586
240,Isle of Wight East,37.063929
280,Louth and Horncastle,36.658838
421,South Devon,36.285482


### 5. Retrieve housing-tenure observations

Housing tenure is included as a socioeconomic predictor of turnout.

The ONS Census API is searched for a tenure classification that can be used to
construct the percentage of households that are owner-occupied, providing a
measure of housing and socioeconomic structure at constituency level.



In [37]:
url = f"{BASE_URL}/population-types"

response = requests.get(
    url,
    params={"limit": 500},
    timeout=30
)
response.raise_for_status()

population_types = response.json()

microdata_populations = [
    item for item in population_types["items"]
    if item.get("type") == "microdata"
]

[(item["name"], item["label"]) for item in microdata_populations]

[('HH', 'All households'),
 ('HRP', 'All Household Reference Persons'),
 ('UR_CE', 'All usual residents in communal establishments'),
 ('UR_HH', 'All usual residents in households'),
 ('UR', 'All usual residents')]

In [38]:
[
    (item["name"], item["label"])
    for item in microdata_populations
    if "household" in item["label"].lower()
]

[('HH', 'All households'),
 ('HRP', 'All Household Reference Persons'),
 ('UR_HH', 'All usual residents in households')]

In [39]:
for item in microdata_populations:
    if "household" in item["label"].lower():

        url = (
            f"{BASE_URL}/population-types/"
            f"{item['name']}/area-types"
        )

        response = requests.get(
            url,
            params={"limit": 500},
            timeout=30
        )
        response.raise_for_status()

        area_types = response.json()["items"]

        if any(area["id"] == "p19wpc" for area in area_types):
            print(item["name"], "-", item["label"])

HH - All households
HRP - All Household Reference Persons
UR_HH - All usual residents in households


In [40]:
HOUSEHOLD_POPULATION = "HH"

url = (
    f"{BASE_URL}/population-types/"
    f"{HOUSEHOLD_POPULATION}/area-types"
)

response = requests.get(
    url,
    params={"limit": 500},
    timeout=30
)
response.raise_for_status()

household_area_types = response.json()["items"]

any(area["id"] == "p19wpc" for area in household_area_types)

True

In [41]:
url = (
    f"{BASE_URL}/population-types/"
    f"{HOUSEHOLD_POPULATION}/dimensions"
)

response = requests.get(
    url,
    params={"q": "tenure"},
    timeout=30
)
response.raise_for_status()

tenure_dimensions = response.json()
tenure_dimensions

{'limit': 30,
 'offset': 0,
 'count': 1,
 'total_count': 45,
 'items': [{'id': 'hh_tenure_9a',
   'label': 'Tenure of household (9 categories)',
   'description': 'Whether a household owns or rents the accommodation that it occupies.\n\nOwner-occupied accommodation can be:\n\n* owned outright, which is where the household owns all of the accommodation\n* with a mortgage or loan\n* part-owned on a shared ownership scheme\n\n Rented accommodation can be:\n\n* private rented, for example, rented through a private landlord or letting agent\n* social rented through a local council or housing association\n\nThis information is not available for household spaces with no usual residents.',
   'total_count': 9,
   'quality_statement_text': 'There is evidence of people incorrectly identifying their type of landlord as ”Council or local authority” or “Housing association”. You should add these two categories together when analysing data that uses this variable.'}]}

#### Identified the household population and tenure variable

Housing tenure is defined at the household level, so it cannot be retrieved from the `UR` population type used for resident-level variables such as age and qualifications.

The ONS API was therefore queried for available microdata population types. `HH` (`All households`) was selected because tenure describes whether a household owns or rents its accommodation.

The household population was then checked to ensure that it supports the post-2019 Westminster constituency geography (`p19wpc`) used throughout the analysis. The tenure dimension `hh_tenure_9a` was identified from the available household-level Census variables.

ONS advises combining the `Council or local authority` and `Housing association` categories when analysing social renting, due to known misclassification between the two.

#### Retrieve housing-tenure observations

The `hh_tenure_9a` dimension is retrieved for all 575 constituencies using the
`HH` (`All households`) population type. The returned categories are inspected
before constructing any housing-tenure predictors.

In [42]:
tenure_dimension = "hh_tenure_9a"

url = f"{BASE_URL}/population-types/HH/census-observations"

params = {
    "area-type": "p19wpc",
    "dimensions": tenure_dimension
}

response = requests.get(
    url,
    params=params,
    timeout=60
)

response.raise_for_status()

tenure_data = response.json()

In [43]:
print("Total observations:", tenure_data["total_observations"])
print("Total areas:", tenure_data["total_areas"])
print("Areas returned:", tenure_data["areas_returned"])

assert tenure_data["total_observations"] == 575 * 9
assert tenure_data["total_areas"] == 575
assert tenure_data["areas_returned"] == 575

Total observations: 5175
Total areas: 575
Areas returned: 575


In [44]:
tenure_long = flatten_census_response(
    tenure_data,
    dimension_id="hh_tenure_9a",
    value_name="tenure"
)

tenure_long.head()

,constituency_code,constituency_name,tenure_code,tenure,count
0,E14001063,Aldershot,-8,Does not apply,0
1,E14001063,Aldershot,0,Owned: Owns outright,12789
2,E14001063,Aldershot,1,Owned: Owns with a mortgage or loan,16373
3,E14001063,Aldershot,2,Shared ownership: Shared ownership,806
4,E14001063,Aldershot,3,Social rented: Rents from council or Local Aut...,1014


In [45]:
tenure_long[
    ["tenure_code", "tenure"]
].drop_duplicates().sort_values("tenure_code")

,tenure_code,tenure
0,-8,Does not apply
1,0,Owned: Owns outright
2,1,Owned: Owns with a mortgage or loan
3,2,Shared ownership: Shared ownership
4,3,Social rented: Rents from council or Local Aut...
5,4,Social rented: Other social rented
6,5,Private rented: Private landlord or letting ag...
7,6,Private rented: Other private rented
8,7,Lives rent free


In [46]:
tenure_long.groupby("constituency_code").size().value_counts()

9    575
Name: count, dtype: int64

#### Construct housing-tenure predictors

Two housing-tenure measures are constructed:

- the percentage of applicable households that are owner-occupied, including
  outright ownership, ownership with a mortgage or loan, and shared ownership;
- the percentage of applicable households that are socially rented.

The `Does not apply` category is excluded from the denominator. ONS recommends
combining the two social-rented categories because respondents may
misclassify council and housing-association landlords.

In [47]:
applicable_tenure = tenure_long[
    tenure_long["tenure_code"] != "-8"
].copy()

tenure_totals = (
    applicable_tenure
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("tenure_total")
)

owner_occupied = (
    applicable_tenure[
        applicable_tenure["tenure_code"].isin(["0", "1", "2"])
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("owner_occupied_count")
)

social_rented = (
    applicable_tenure[
        applicable_tenure["tenure_code"].isin(["3", "4"])
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("social_rented_count")
)

tenure_features = (
    pd.concat(
        [tenure_totals, owner_occupied, social_rented],
        axis=1
    )
    .reset_index()
)

tenure_features["owner_occupied_pct"] = (
    tenure_features["owner_occupied_count"]
    / tenure_features["tenure_total"]
    * 100
)

tenure_features["social_rented_pct"] = (
    tenure_features["social_rented_count"]
    / tenure_features["tenure_total"]
    * 100
)

In [48]:
assert len(tenure_features) == 575
assert tenure_features["constituency_code"].is_unique
assert tenure_features["owner_occupied_pct"].between(0, 100).all()
assert tenure_features["social_rented_pct"].between(0, 100).all()

tenure_features[
    ["owner_occupied_pct", "social_rented_pct"]
].describe()

,owner_occupied_pct,social_rented_pct
count,575.000000,575.000000
mean,63.090328,16.803375
std,11.698170,6.844751
min,22.597279,5.412283
25%,57.573309,11.958912
50%,66.017033,14.959219
75%,71.323900,19.962328
max,82.548809,47.592068


In [49]:
tenure_features.nlargest(
    10, "owner_occupied_pct"
)[["constituency_name", "owner_occupied_pct"]]

,constituency_name,owner_occupied_pct
95,Cheadle,82.548809
400,Sefton Central,82.513703
374,Rayleigh and Wickford,82.294458
456,Stockton West,80.827818
380,Ribble Valley,80.696119
91,Castle Point,80.491644
542,York Outer,80.344604
526,Wirral West,80.209383
301,Mid Leicestershire,79.598958
405,Sheffield Hallam,79.253956


In [50]:
tenure_features.nlargest(
    10, "social_rented_pct"
)[["constituency_name", "social_rented_pct"]]

,constituency_name,social_rented_pct
358,Peckham,47.592068
496,Vauxhall and Camberwell Green,45.081627
197,Hackney South and Shoreditch,44.599959
227,Holborn and St Pancras,43.321609
243,Islington South and Finsbury,41.982925
23,Bethnal Green and Stepney,40.792096
372,Queen's Park and Maida Vale,38.356935
242,Islington North,37.697367
22,Bermondsey and Old Southwark,37.217692
196,Hackney North and Stoke Newington,37.059117


#### Validated housing-tenure measures

The resulting tenure measures show substantial variation across constituencies and produce geographically plausible extremes. High owner-occupation rates are concentrated in suburban and outer-urban constituencies, while the highest social-rented shares are concentrated in inner London.

These checks provide a basic validation that the tenure categories have been aggregated correctly before the variables are incorporated into the final analytical dataset.

### 6. Retrieve labour-market observations

Labour-market conditions are included as an additional socioeconomic predictor of turnout.

Because economic-activity measures are defined for an age-specific population rather than for all usual residents, the relevant ONS microdata population type is identified and checked for compatibility with the post-2019 Westminster constituency geography before retrieving the Census variable.

In [51]:
url = f"{BASE_URL}/population-types/UR/dimensions"

response = requests.get(
    url,
    params={"q": "economic activity"},
    timeout=30
)
response.raise_for_status()

economic_activity_dimensions = response.json()
economic_activity_dimensions

{'limit': 30,
 'offset': 0,
 'count': 2,
 'total_count': 44,
 'items': [{'id': 'activity_last_week',
   'label': 'Economic activity status last week (8 categories)',
   'description': 'This variable presents the same information as economic activity status, but in a slightly different way. \n\nPeople aged 16 years and over were classified as “working” if they were economically active and in employment between 15 and 21 March 2021. This category is referred to as “employed” when looking in the higher-level (3 category) version of this variable in the Flexible Table Builder. \n\n“Unemployed” refers to people who said they were out of work during the same period, but were either looking for work and could start within two weeks, or were waiting to start a job that had been offered and accepted. \n\n“Economically inactive” refers to people aged 16 years and over who did not have a job between 15 to 21 March 2021, and who had not looked for work between 22 February to 21 March 2021, or coul

#### Selected labour-market measure

The `economic_activity_status_12a` dimension is used because it distinguishes
employment, unemployment and economic inactivity while separately identifying
full-time students.

For the analysis, student and non-student categories will be combined within
employment and unemployment. This allows an unemployment rate to be calculated
as the percentage of economically active residents aged 16 and over who were
unemployed.

ONS advises caution when interpreting Census 2021 labour-market measures because
the Census took place during a period of substantial labour-market disruption.

In [52]:
economic_activity_dimension = "economic_activity_status_12a"

url = f"{BASE_URL}/population-types/UR/census-observations"

params = {
    "area-type": "p19wpc",
    "dimensions": economic_activity_dimension
}

response = requests.get(
    url,
    params=params,
    timeout=60
)

response.raise_for_status()

economic_activity_data = response.json()

In [53]:
print("Total observations:", economic_activity_data["total_observations"])
print("Total areas:", economic_activity_data["total_areas"])
print("Areas returned:", economic_activity_data["areas_returned"])

assert economic_activity_data["total_observations"] == 575 * 12
assert economic_activity_data["total_areas"] == 575
assert economic_activity_data["areas_returned"] == 575

Total observations: 6900
Total areas: 575
Areas returned: 575


In [54]:
economic_activity_long = flatten_census_response(
    economic_activity_data,
    dimension_id="economic_activity_status_12a",
    value_name="economic_activity"
)

In [55]:
for code, label in (
    economic_activity_long[
        ["economic_activity_code", "economic_activity"]
    ]
    .drop_duplicates()
    .sort_values("economic_activity_code")
    .itertuples(index=False, name=None)
):
    print(code, "-", label)

-8 - Does not apply
1 - Economically active (excluding full-time students): In employment: Employee
10 - Economically inactive: Long-term sick or disabled
11 - Economically inactive: Other
2 - Economically active (excluding full-time students): In employment: Self-employed with employees
3 - Economically active (excluding full-time students): In employment: Self-employed without employees
4 - Economically active (excluding full-time students): Unemployed: Seeking work or waiting to start a job already obtained: Available to start working within 2 weeks
5 - Economically active and a full-time student: In employment
6 - Economically active and a full-time student: Unemployed: Seeking work or waiting to start a job already obtained: Available to start working within 2 weeks
7 - Economically inactive: Retired
8 - Economically inactive: Student
9 - Economically inactive: Looking after home or family


In [56]:
economic_activity_long.groupby(
    "constituency_code"
).size().value_counts()

12    575
Name: count, dtype: int64

#### Construct unemployment rate

Unemployment is measured as the percentage of economically active residents
aged 16 and over who were unemployed.

Both full-time students and non-students are included where they were
economically active. Economically inactive residents are excluded from the
denominator, so the measure represents an unemployment rate rather than the
share of the total population that was unemployed.

Census 2021 labour-market data should be interpreted cautiously because the
Census was conducted during an unusual period of labour-market disruption.

In [57]:
economically_active_codes = ["1", "2", "3", "4", "5", "6"]
unemployed_codes = ["4", "6"]

economically_active = (
    economic_activity_long[
        economic_activity_long["economic_activity_code"]
        .isin(economically_active_codes)
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("economically_active_count")
)

unemployed = (
    economic_activity_long[
        economic_activity_long["economic_activity_code"]
        .isin(unemployed_codes)
    ]
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("unemployed_count")
)

economic_activity_features = (
    pd.concat(
        [economically_active, unemployed],
        axis=1
    )
    .reset_index()
)

economic_activity_features["unemployment_rate"] = (
    economic_activity_features["unemployed_count"]
    / economic_activity_features["economically_active_count"]
    * 100
)

In [58]:
economic_activity_features.nlargest(
    10, "unemployment_rate"
)[
    ["constituency_name", "unemployment_rate"]
]

,constituency_name,unemployment_rate
33,Birmingham Ladywood,16.248227
290,Manchester Rusholme,14.864471
35,Birmingham Perry Barr,13.005655
256,Leeds Central and Headingley,13.003496
275,Liverpool Riverside,12.930446
404,Sheffield Central,11.231466
57,Bradford West,11.178444
32,Birmingham Hodge Hill and Solihull North,11.068384
264,Leicester South,10.869731
304,Middlesbrough and Thornaby East,10.689904


In [59]:
economic_activity_features.nsmallest(
    10, "unemployment_rate"
)[
    ["constituency_name", "unemployment_rate"]
]

,constituency_name,unemployment_rate
412,Skipton and Ripon,2.889564
517,Westmorland and Lonsdale,2.895560
519,Wetherby and Easingwold,2.938715
528,Witney,3.026558
380,Ribble Valley,3.053329
432,South West Devon,3.103616
542,York Outer,3.125479
481,Thirsk and Malton,3.129185
482,Thornbury and Yate,3.149417
134,Didcot and Wantage,3.150202


#### Validated unemployment measure

The unemployment rate shows plausible variation across constituencies. The highest rates are concentrated in relatively deprived urban constituencies, while the lowest rates are generally found in more affluent areas.

This provides a basic substantive check that the economically active denominator and unemployment categories have been combined correctly. The measure is retained as the labour-market predictor for the analysis.

### 7. Construct population-density data

The ONS Population Density topic-summary dataset does not expose the post-2019
Westminster constituency geography required for this analysis.

Population density is therefore constructed directly using two official ONS
sources. Total usual-resident population is obtained by summing the Census age
categories already retrieved for each constituency. Constituency land area is
obtained from the ONS Geography July 2024 Westminster Parliamentary Constituency
boundary dataset.

Density is calculated as usual residents per square kilometre.

In [60]:
BOUNDARY_URL = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Westminster_Parliamentary_Constituencies_July_2024_Boundaries_UK_BGC/"
    "FeatureServer/0/query"
)

params = {
    "where": "1=1",
    "outFields": "PCON24CD,PCON24NM,Shape__Area",
    "returnGeometry": "false",
    "f": "json"
}

response = requests.get(
    BOUNDARY_URL,
    params=params,
    timeout=60
)

response.raise_for_status()

boundary_data = response.json()

In [61]:
boundary_df = pd.DataFrame(
    feature["attributes"]
    for feature in boundary_data["features"]
).rename(columns={
    "PCON24CD": "constituency_code",
    "PCON24NM": "constituency_name",
    "Shape__Area": "area_sq_m"
})

boundary_df.shape

(650, 3)

In [62]:
boundary_ew = (
    boundary_df[
        boundary_df["constituency_code"].isin(election_codes)
    ]
    .copy()
)

assert len(boundary_ew) == 575
assert boundary_ew["constituency_code"].is_unique
assert set(boundary_ew["constituency_code"]) == election_codes

In [63]:
census_population = (
    age_long
    .groupby(["constituency_code", "constituency_name"])["count"]
    .sum()
    .rename("census_population")
    .reset_index()
)

assert len(census_population) == 575
assert census_population["constituency_code"].is_unique

In [64]:
density_features = (
    census_population
    .merge(
        boundary_ew[
            ["constituency_code", "area_sq_m"]
        ],
        on="constituency_code",
        how="left",
        validate="one_to_one"
    )
)

assert density_features["area_sq_m"].notna().all()

density_features["area_sq_km"] = (
    density_features["area_sq_m"] / 1_000_000
)

density_features["population_density"] = (
    density_features["census_population"]
    / density_features["area_sq_km"]
)

In [65]:
assert len(density_features) == 575
assert density_features["population_density"].gt(0).all()

density_features["population_density"].describe()

count      575.000000
mean      2283.315711
std       2904.953688
min         29.710433
25%        311.383165
50%       1083.930213
75%       3279.353056
max      18625.367554
Name: population_density, dtype: float64

In [66]:
density_features.nlargest(
    10, "population_density"
)[
    ["constituency_name", "population_density"]
]

,constituency_name,population_density
23,Bethnal Green and Stepney,18625.367554
372,Queen's Park and Maida Vale,16306.987157
367,Poplar and Limehouse,15193.040779
242,Islington North,14880.375418
243,Islington South and Finsbury,14376.376893
496,Vauxhall and Camberwell Green,14246.441806
197,Hackney South and Shoreditch,13951.695873
112,Clapham and Brixton Hill,13790.066888
358,Peckham,13328.064345
150,East Ham,13098.125079


In [67]:
density_features.nsmallest(
    10, "population_density"
)[
    ["constituency_name", "population_density"]
]

,constituency_name,population_density
547,"Brecon, Radnor and Cwm Tawe",29.710433
222,Hexham,33.172594
558,Dwyfor Meirionnydd,36.735507
517,Westmorland and Lonsdale,37.663567
555,Ceredigion Preseli,38.347758
564,Montgomeryshire and Glyndwr,40.971902
334,North Northumberland,42.848592
549,Caerfyrddin,46.478714
412,Skipton and Ripon,48.000784
481,Thirsk and Malton,48.766811


#### Validated population-density measure

Population density aligns with the expected geographic pattern. The highest-density constituencies are all in London, while the lowest-density constituencies are in rural areas.

This provides a substantive check that Census population totals and constituency areas have been combined correctly.

### 8. Building final merged dataset

The census features have been retrieved/derived, so I am ready to construct our final analytical dataset. 

The Census predictors are merged with the processed 2024 election results using
the official constituency code.

Because the Census and election datasets use the same post-2019 Westminster
constituency geography, the merge can be performed directly without fuzzy name
matching or geographic crosswalks.

#### Building census features table

In [68]:
census_features = (
    qualification_features[
        ["constituency_code", "level4_plus_pct"]
    ]
    .merge(
        age_features[
            ["constituency_code", "age_18_34_pct", "age_65_plus_pct"]
        ],
        on="constituency_code",
        validate="one_to_one"
    )
    .merge(
        tenure_features[
            ["constituency_code", "owner_occupied_pct", "social_rented_pct"]
        ],
        on="constituency_code",
        validate="one_to_one"
    )
    .merge(
        economic_activity_features[
            ["constituency_code", "unemployment_rate"]
        ],
        on="constituency_code",
        validate="one_to_one"
    )
    .merge(
        density_features[
            ["constituency_code", "population_density"]
        ],
        on="constituency_code",
        validate="one_to_one"
    )
)

In [69]:
assert len(census_features) == 575
assert census_features["constituency_code"].is_unique
assert census_features.notna().all().all()

census_features.describe()

,level4_plus_pct,age_18_34_pct,age_65_plus_pct,owner_occupied_pct,social_rented_pct,unemployment_rate,population_density
count,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000
mean,33.571435,27.088076,23.855425,63.090328,16.803375,5.598011,2283.315711
std,9.020211,7.465308,6.387967,11.698170,6.844751,1.952120,2904.953688
min,18.084787,15.880229,6.235623,22.597279,5.412283,2.889564,29.710433
25%,27.041060,22.188620,19.963960,57.573309,11.958912,4.136567,311.383165
50%,32.048674,25.272725,24.350059,66.017033,14.959219,5.084168,1083.930213
75%,37.615543,29.770854,28.237907,71.323900,19.962328,6.529543,3279.353056
max,65.221223,66.517514,41.174165,82.548809,47.592068,16.248227,18625.367554


#### Assemble analytical dataset



In [70]:
analysis_data = (
    election
    .merge(
        census_features,
        on="constituency_code",
        how="left",
        validate="one_to_one"
    )
)

assert len(analysis_data) == 575
assert analysis_data["constituency_code"].is_unique
assert analysis_data.notna().all().all()

analysis_data.shape

(575, 18)

In [71]:
predictor_cols = [
    "level4_plus_pct",
    "age_18_34_pct",
    "age_65_plus_pct",
    "owner_occupied_pct",
    "social_rented_pct",
    "unemployment_rate",
    "population_density"
]

analysis_data[
    ["turnout_pct"] + predictor_cols
].describe()

,turnout_pct,level4_plus_pct,age_18_34_pct,age_65_plus_pct,owner_occupied_pct,social_rented_pct,unemployment_rate,population_density
count,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000
mean,60.116025,33.571435,27.088076,23.855425,63.090328,16.803375,5.598011,2283.315711
std,7.010406,9.020211,7.465308,6.387967,11.698170,6.844751,1.952120,2904.953688
min,40.251763,18.084787,15.880229,6.235623,22.597279,5.412283,2.889564,29.710433
25%,55.005884,27.041060,22.188620,19.963960,57.573309,11.958912,4.136567,311.383165
50%,60.674673,32.048674,25.272725,24.350059,66.017033,14.959219,5.084168,1083.930213
75%,65.747235,37.615543,29.770854,28.237907,71.323900,19.962328,6.529543,3279.353056
max,75.474101,65.221223,66.517514,41.174165,82.548809,47.592068,16.248227,18625.367554


In [72]:
assert set(analysis_data["constituency_code"]) == election_codes
assert analysis_data["constituency_code"].nunique() == 575

### 9. Export analytical dataset

The completed analytical dataset contains 575 constituencies in England and Wales
and combines 2024 general-election turnout with seven demographic and socioeconomic
predictors derived from Census 2021 and ONS geography data.

All datasets are joined using official constituency codes, with complete one-to-one
coverage across the analytical sample. The resulting dataset contains no missing
values in the outcome or predictor variables and is exported for exploratory
analysis and modelling.

The processed dataset is exported for exploratory analysis and modelling in the
subsequent notebooks.

In [73]:
analysis_data.to_csv(
    "../data/processed/analysis_data.csv",
    index=False
)